In [ ]:
import time
# autoreload
%load_ext autoreload
%autoreload 2

#from scipy import signal
#from scipy import interpolate
#from scipy import ndimage
import numpy as np
#import pycatch22 
#from sktime.transformations.panel import catch22
#import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
import random
load_dotenv = dotenv.load_dotenv('../.env')

# load local library
from timex import clustering
from timex import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

import datetime
from time import sleep




In [ ]:
AKI_PATH = os.environ['AKI_PATH_NEW']
os.chdir(AKI_PATH)

In [ ]:
AKI_PATH


In [ ]:
os.listdir('.')

In [ ]:
ts_data = pd.read_parquet(os.path.join(AKI_PATH, 'cleaned_DV_LCMM_data.parquet'))
ts_data = ts_data.rename(columns={"Time_since_index_FU_days": "Time_days"})
ts_data.ID = ts_data.ID.astype('int64')
ts_data['dataset_nr'] = ts_data['dataset_nr'].fillna(-1)


In [ ]:
ts_data.groupby('dataset_nr').ID.nunique(), ts_data.ID.nunique()

In [ ]:
MIN_TIME = 365 # days
MAX_TIME = 365*10 # days
MIN_MEAS_COUNT = 3 # measurements
INTERP_RES = 90
SMOOTHING_WINDOW = 4 # in days: 4 * INTERP_RES = 360
SMOOTHING_TYPE = 'gaussian_kernel'  # 'gaussian_kernel' or 'rolling_mean'
META_KEYS = ['ID', 'Time_days']
RAW_VAL_COL = 'eGFRcr_CKDEpi2009'
INT_VAL_COL = 'eGFR_int'
SM30_VAL_COL = 'eGFR_SW30'
SM365_VAL_COL = 'eGFR_SW365'

In [ ]:
ts_data_df = ts_data[['ID', 'Time_days', 'eGFRcr_CKDEpi2009']].dropna(subset=['eGFRcr_CKDEpi2009'])

In [ ]:
ts_clusterer = clustering.CrossSectionalClustering(smoothing=True, 
                                                   smoothing_type=SMOOTHING_TYPE,
                                                   smoothing_window_size=SMOOTHING_WINDOW,
                                                   n_skip=3,
                                                   interpolation=True, 
                                                   interpolation_resolution=INTERP_RES,
                                                   interpolation_keep_init=True,
                                                   min_measurements_per_id=MIN_MEAS_COUNT, 
                                                   min_time=MIN_TIME,
                                                   max_time=MAX_TIME,
                                                   n_clusters=3, 
                                                   id_column='ID', 
                                                   time_column='Time_days',
                                                   feature_columns=[RAW_VAL_COL],
                                                   imputation_method='knn',
                                                   cross_standardisation=True,
                                                   normalise_timeseries= "group",
                                                   normalisation_method="standard",
                                                   add_ts_meta=False,
                                                   verbose=True)

In [ ]:
ts_clusterer.fit(ts_data_df)

In [42]:
ts_temp = ts_clusterer.ts_filtered
ts_temp.loc[ts_temp.ID==3565782].head()

,ID,Time_days,eGFRcr_CKDEpi2009
1342,3565782,0.000000,108.503346
1347,3565782,4.102778,90.935916
1354,3565782,10.097917,88.845981
1360,3565782,15.073611,80.437337
1368,3565782,22.159028,84.919419


In [43]:
ts_temp = ts_clusterer.ts_interpolated
ts_temp.loc[ts_temp.ID==3565782].head()

,ID,Time_days,eGFRcr_CKDEpi2009
0,3565782,0,108.503346
1,3565782,90,109.304904
2,3565782,180,138.094540
3,3565782,270,133.880216
4,3565782,360,123.972524


In [44]:
ts_temp = ts_clusterer.ts_smoothed
ts_temp.loc[ts_temp.ID==3565782].head()


,ID,Time_days,eGFRcr_CKDEpi2009
0,3565782,0,108.503346
1,3565782,90,109.304904
2,3565782,180,138.094540
3,3565782,270,NaN
4,3565782,360,NaN


In [ ]:
ts_temp = ts_clusterer.ts_normalized
ts_temp.loc[ts_temp.ID==3565782].head()
